In [ ]:
# This example uses the GQL GraphQL client library.
#
# To install: pip3 install gql
#
# GQL is one popular Python GraphQL client, but there are others.
# See https://graphql.org/community/tools-and-libraries/?tags=python_client

from gql import gql, Client
from gql.transport.aiohttp import AIOHTTPTransport

transport = AIOHTTPTransport(url="https://gnomad.broadinstitute.org/api")
client = Client(transport=transport, fetch_schema_from_transport=True)

# For brevity, and to keep the focus on the Python code, we don't include every
# field from the raw query here.

query = gql(
    """
    query VariantsInGene {
      gene(gene_symbol: "MAP7D1", reference_genome: GRCh38) {
        variants(dataset: gnomad_r4) {
            variant_id
            pos
            rsids
            transcript_id
            transcript_version
            hgvs
            hgvsc
            hgvsp
            flags
            consequence
            exome {
                ac
                ac_hemi
                ac_hom
                an
                af
                populations {
                    id
                    ac
                    an
                    ac_hemi
                    ac_hom
                }
            }
        }
      }
    }
"""
)

# Execute the query on the transport
result = await client.execute_async(query)
print(list(result.keys()))
print(list(result["gene"].keys()))
for el in result["gene"]["variants"]:
    if el["pos"] == 36176314:
        print(el)
# print(result["gene"]["variants"])

In [ ]:

# print(test_gnomad.head())
# print(test_gnomad.columns)
# print(test_gnomad.head())

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pyranges as pr   # BEST library for interval overlaps
import json
from Bio.Seq import Seq
###############################################
# INPUT FILE PATHS
###############################################

# Replace with your files:
pos_bed = "/mnt/d/phd/scripts/14_gnomAD_conservation_proj/pos_RGmotifs_coordinates.bed"
neg_bed = "/mnt/d/phd/scripts/14_gnomAD_conservation_proj/neg_RGmotifs_coordinates.bed"

# pos_variants_file = "positive_variants.tsv"
# neg_variants_file = "negative_variants.tsv"

###############################################
# LOAD BED FILES
###############################################

# BED columns: CHROM, START, END, STRAND, NAME, ...
# We need CHROM, START, END, NAME
bed_cols = ["CHROM", "START", "END", "STRAND", "NAME"]

pos_regions = pd.read_csv(pos_bed, sep="\t", header=None, names=bed_cols+list(range(7)))
neg_regions = pd.read_csv(neg_bed, sep="\t", header=None, names=bed_cols+list(range(7)))

# Select only the important columns
pos_regions = pos_regions[["CHROM", "START", "END", "STRAND", "NAME"]].copy()
neg_regions = neg_regions[["CHROM", "START", "END", "STRAND", "NAME"]].copy()

# Add group label
pos_regions["group"] = "positive"
neg_regions["group"] = "negative"

# Combine region tables
regions = pd.concat([pos_regions, neg_regions], ignore_index=True)

###############################################
# PARSE REGION_ID FROM COLUMN 5
###############################################

# Example NAME format:
# Q12774_858_868_chr7_144365241_144365274_+

def parse_region_id(name):
    protein = name.split("_")[0]
    rg_start = name.split("_")[1]
    rg_end = name.split("_")[2]
    region_id = f"{protein}_{rg_start}_{rg_end}"
    return region_id

regions["region_id"] = regions["NAME"].apply(parse_region_id)

###############################################
# LOAD VARIANTS
###############################################

# df_pos = pd.read_csv(pos_variants_file, sep="\t")
df_pos = pd.read_csv("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/output/combined_joint_variants_pos.tsv", sep="\t")
# df_neg = pd.read_csv(neg_variants_file, sep="\t")
df_neg = pd.read_csv("/mnt/d/phd/scripts/14_gnomAD_conservation_proj/output/combined_joint_variants_neg.tsv", sep="\t")

df_pos["group"] = "positive"
df_neg["group"] = "negative"

variants = pd.concat([df_pos, df_neg], ignore_index=True)

###############################################
# PREPARE PANDAS → PYRANGES OBJECTS
###############################################

# Convert BED regions to pyranges
pr_regions = pr.PyRanges(regions.rename(columns={"CHROM":"Chromosome", "START": "Start", "END": "End"}))

# Convert variants into intervals (POS → POS+1)
variants_interval = variants.rename(columns={"CHROM":"Chromosome", "POS":"Start"})
variants_interval["End"] = variants_interval["Start"] + 1
pr_variants = pr.PyRanges(variants_interval)

###############################################
# OVERLAP: ASSIGN VARIANTS TO REGIONS
###############################################

overlap = pr_variants.join(pr_regions)

df_overlap = overlap.as_df()

# Now df_overlap contains all matched variants with:
# Chromosome, Start, End, REF, ALT, AF, region_id, group, etc.

###############################################
# COMPUTE VARIANT METRICS PER REGION
###############################################

pcs_list = []

for region_id, group_df in df_overlap.groupby("region_id"):

    # Extract region metadata
    region_meta = regions[regions["region_id"] == region_id].iloc[0]
    r_start = region_meta["START"]
    r_end = region_meta["END"]
    r_group = region_meta["group"]

    region_length = r_end - r_start  # BED end is non-inclusive

    # ABSOLUTE number of variant-bearing positions
    unique_positions = group_df["Start"].nunique()

    # NORMALIZED variant density
    variant_density = unique_positions / region_length

    pcs_list.append({
        "region_id": region_id,
        "group": r_group,
        "region_length": region_length,
        "variant_positions": unique_positions,
        "variant_density": variant_density,
        "pcs": 1 - variant_density   # keep PCS if needed
    })

pcs_df = pd.DataFrame(pcs_list)

###############################################
# STATISTICAL TEST (on variant_density)
###############################################

pos_scores = pcs_df[pcs_df["group"] == "positive"]["variant_density"]
neg_scores = pcs_df[pcs_df["group"] == "negative"]["variant_density"]

u_stat, pval = mannwhitneyu(pos_scores, neg_scores, alternative="two-sided")

print(f"Mann–Whitney U test (variant_density):")
print(f"U = {u_stat:.3f}, p = {pval:.4e}")

###############################################
# VISUALIZATION: BOXPLOT + SCATTER (STRIP)
###############################################

plt.figure(figsize=(9, 5))

# Boxplot of normalized variant density
sns.boxplot(
    data=pcs_df,
    x="group",
    y="variant_density",
    width=0.4,
    showcaps=True,
    boxprops={"zorder": 2},
    showfliers=False
)

# Add scatter points (jittered)
sns.stripplot(
    data=pcs_df,
    x="group",
    y="variant_density",
    size=4,
    alpha=0.6,
    jitter=True,
    color="black"
)

plt.title(f"Variant Density per Region (p = {pval:.2e})")
plt.ylabel("Variant Density = variant_positions / region_length")
plt.xlabel("")
plt.ylim(0,2)
plt.tight_layout()
plt.show()

###############################################
# OPTIONAL: Display summary table
###############################################

print("\nSummary (first 10 rows):")
print(pcs_df.head(10))


In [ ]:
#### now we check number of RGs before and after

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import mannwhitneyu

type_of_AF = "AF_exomes"

# Remove zeros
df_filtered = df_overlap[df_overlap[type_of_AF] > 0]

# Optional: compute statistical test
pos_vals = df_filtered[df_filtered['group_b'] == 'positive'][type_of_AF]
neg_vals = df_filtered[df_filtered['group_b'] == 'negative'][type_of_AF]

stat, p_value = mannwhitneyu(neg_vals, pos_vals ,alternative='two-sided')
print("Negative AF median:", neg_vals.median())
print("Positive AF median:", pos_vals.median())
# print(stat)

# Plot
plt.figure(figsize=(8,6))
sns.boxplot(x='group_b', y=type_of_AF, data=df_filtered)
sns.stripplot(x='group_b', y=type_of_AF, data=df_filtered, color='black', alpha=0.5, jitter=0.2)

plt.yscale('log')  # logarithmic y-axis
plt.ylabel(type_of_AF + '(log scale)')
plt.title('AF_exomes distribution: Positive vs Negative')

# Annotate p-value
x1, x2 = 0, 1  # positions of the two boxes
y, h, col = df_filtered[type_of_AF].max()*1.1, df_filtered[type_of_AF].max()*0.1, 'k'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h*1.2, f'p = {p_value:.3e}', ha='center', color=col)

plt.show()

In [ ]:
df_overlap

In [ ]:
from typing import Literal

VariantType = Literal[
    "silent",
    "missense",
    "nonsense",
    "inframe_insertion",
    "inframe_deletion",
    "complex",
    "frameshift"
]


def classify_variant(before_dna: str, after_dna: str,
                     before_aa: str, after_aa: str) -> VariantType:
    """
    Classify a protein/DNA change into mutation types:
    silent, missense, nonsense, inframe_insertion, 
    inframe_deletion, complex, frameshift.

    Assumes coding DNA and correct translation.
    """
    if after_dna is None:
        return None
    if after_aa is None:
        return None
    if before_aa == after_aa:
        return "silent"

    # 3. Same AA length → could be silent, missense, nonsense, complex
    # Compare residue by residue
    diffs = [(i, a, b) for i, (a, b) in enumerate(zip(before_aa, after_aa), 1) if a != b]

    # 4. If any change introduces a stop codon
    if any(new == "*" for _, _, new in diffs):
        return "nonsense"

    # 5. Single AA change → missense
    if len(diffs) == 1 and len(before_aa)==len(after_aa):
        return "missense"
    # 1. Frameshift check (DNA length change not multiple of 3)
    dna_len_change = len(after_dna) - len(before_dna)
    if dna_len_change % 3 != 0:
        return "frameshift"

    # 2. In-frame insertion or deletion (no frameshift)
    aa_len_change = len(after_aa) - len(before_aa)

    if aa_len_change > 0:
        return "inframe_insertion"
    elif aa_len_change < 0:
        return "inframe_deletion"
    # 6. Multiple AA changes but no frameshift → complex substitution
    return "complex"


print(classify_variant("CCCCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "CCTCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "PRGEAPGPGRRG", "PRGEAPGPGRRG"))
print(classify_variant("CCCCGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "CCCAGCGGGGAAGCTCCTGGCCCCGGGAGACGGGGG", "PRGEAPGPGRRG", "PSGEAPGPGRRG"))
print(classify_variant("AAGCGAGGGAGTGTGAGCAGGGGC", "AAGTGAGGGAGTGTGAGCAGGGGC", "KRGSVSRG", "K*GSVSRG"))
print(classify_variant("TCCCGGGGAGGCGGAGGGGGATCCCGCGGGGGC", "TCCCGGGAGGCGGAGGGGGATCCCGCGGGGGC", "SRGGGGGSRGG", "SREAEGDPAG"))
print(classify_variant("CATCGAGGCGGCGGGGAGCCCCGCGGGGGC", "CATCGAGGCGGCGGGGAGCCCCGCGGG", "HRGGGEPRGG", "HRGGGEPRG"))

# for el in ["CATCGAGGCGGCGGGGAGCCCCGCGGGGGC", "CATCGAGGCGGCGGGGAGCCCCGCGGG"]:
#     print(str(Seq(el).translate()))
#     print(len(el))

In [ ]:
import re
from localcider.sequenceParameters import SequenceParameters as SeqParams

def get_physchem_metrics(before_aa: str, after_aa: str, category: str):
    

    # ---------- per-residue property changes ----------
    # charge_change = 0
    # polarity_change = 0
    # aromatic_change = 0
    # print(before_aa)
    # print(after_aa)
    # print(category)
    if category == "nonsense" or before_aa is None or after_aa is None or '*' in after_aa or after_aa is "":
        return {
        "NCPR_change": None,
        "FCR_change": None,
        "hydropathy_change": None,
        "kappa_change": None,
        "pos_count_change": None,
        "neg_count_change": None,
        "aromaticity_change": None
        }
    # align by min length (simple assumption; this is intentional)
    # L = min(len(before_aa), len(after_aa))

    NCPR_change = SeqParams(after_aa).get_NCPR() - SeqParams(before_aa).get_NCPR()
    FCR_change = SeqParams(after_aa).get_FCR() - SeqParams(before_aa).get_FCR()
    hydropathy_change = SeqParams(after_aa).get_mean_hydropathy() - SeqParams(before_aa).get_mean_hydropathy()
    kappa_change = SeqParams(after_aa).get_kappa() - SeqParams(before_aa).get_kappa()
    pos_count_change = SeqParams(after_aa).get_countPos() - SeqParams(before_aa).get_countPos()
    neg_count_change = SeqParams(after_aa).get_countNeg() - SeqParams(before_aa).get_countNeg()
    aromaticity_change =    ((SeqParams(after_aa).get_amino_acid_fractions()["Y"] + 
                            SeqParams(after_aa).get_amino_acid_fractions()["F"] + 
                            SeqParams(after_aa).get_amino_acid_fractions()["W"]) - 
                            (SeqParams(before_aa).get_amino_acid_fractions()["Y"] + 
                            SeqParams(before_aa).get_amino_acid_fractions()["F"] + 
                            SeqParams(before_aa).get_amino_acid_fractions()["W"] ))


    return {

        "NCPR_change": NCPR_change,
        "FCR_change": FCR_change,
        "hydropathy_change": hydropathy_change,
        "kappa_change": kappa_change,
        "pos_count_change": pos_count_change,
        "neg_count_change": neg_count_change,
        "aromaticity_change": aromaticity_change
    }


def count_RG_positions(seq):
    """Return all start positions of 'RG' motifs (0-based)."""
    return [m.start() for m in re.finditer("RG", seq)]


def rg_change_from_category(category, before_aa, after_aa,
                            mut_pos_dna, ref_dna, alt_dna):
    """
    category: one of
        'silent', 'missense', 'inframe_indel', 'frameshift',
        'nonsense', 'noncoding'
    before_aa: AA sequence before mutation
    after_aa: AA sequence after mutation (None if not applicable)
    mut_pos_dna: 1-based genomic DNA start position of the REF allele
    ref_dna, alt_dna: provided but only needed for frameshift logic

    Returns:
        {
            'rg_before': [...],
            'rg_after': [... or None],
            'gained': int,
            'lost': int,
            'unchanged': int
        }
    """
    # if category == None:
    #     return {
    #         'category': category,
    #         'rg_before': rg_before,
    #         'rg_after': rg_after,
    #         'gained': len(new),
    #         'lost': len(lost),
    #         'unchanged': len(unchanged)
    #     }


    # Count RG before
    rg_before = count_RG_positions(before_aa)

    # # ==============================================================
    # # CASE 1 — Noncoding variants
    # # ==============================================================
    # if category == "noncoding":
    #     return {
    #         'category': category,
    #         'rg_before': rg_before,
    #         'rg_after': rg_before,
    #         'gained': 0,
    #         'lost': 0,
    #         'unchanged': len(rg_before)
    #     }

    # ==============================================================
    # CASE 2 — Silent variants
    # (AA sequences identical)
    # ==============================================================
    if category in ("silent", None):
        return {
            # 'category': category,
            'rg_before': rg_before,
            'rg_after': rg_before,
            'gained': 0,
            'lost': 0,
            'unchanged': len(rg_before)
        }

    # ==============================================================
    # CASE 3 — Missense / In-frame Indels / Nonsense
    # (Direct AA comparison)
    # ==============================================================
    if category in ("missense", "nonsense",  "inframe_insertion", "inframe_deletion"):
        rg_after = count_RG_positions(after_aa)

        lost = len([pos for pos in rg_before if pos not in rg_after])
        gained = len([pos for pos in rg_after if pos not in rg_before])
        unchanged = len(rg_before) - lost

        return {
            # 'category': category,
            'rg_before': rg_before,
            'rg_after': rg_after,
            'gained': gained,
            'lost': lost,
            'unchanged': unchanged
        }

    # ==============================================================
    # CASE 4 — Frameshift
    # Only RG motifs upstream of the mutation position remain valid.
    # ==============================================================
    if category == "frameshift":
        aa_mut_pos = mut_pos_dna // 3
        # print(mut_pos_dna)
        # print(aa_mut_pos)
        rg_after = count_RG_positions(after_aa)

        # RGs before the mutation are unchanged
        unchanged = [pos for pos in rg_before if pos < aa_mut_pos]

        # RGs in the original sequence that overlap or are after the mutation are lost
        lost = [pos for pos in rg_before if pos >= aa_mut_pos]

        # RGs in the mutated sequence that are at or after the mutation are new
        new = [pos for pos in rg_after if pos >= aa_mut_pos]

        return {
            'rg_before': rg_before,
            'rg_after': rg_after,
            'gained': len(new),
            'lost': len(lost),
            'unchanged': len(unchanged)
        }

    # ==============================================================
    # Unknown category
    # ==============================================================
    # print(category)
    raise ValueError(f"Unknown category: {category}")

print(rg_change_from_category("missense", "PRGEAPGPGRRG", "LRGEAPGPGRRG", 1, "C", "T"))
print(rg_change_from_category("missense", "PRGEAPGPGRRG", "PSGEAPGPGRRG", 3, "C", "A"))
print(rg_change_from_category("frameshift", "PRGEAPGPGRRG", "PRGKLLAPGDG", 4, "GC", "G"))
print(rg_change_from_category("inframe_insertion", "SRGGGGGSRGG", "SRGGGGGSRGGGSRGG",9, "G", "GGCGGAGGGGGATCCC"))
# print(rg_change_from_category("missense", "PRGEAPGPGRRG", "PSGEAPGPGRRG", 3, "C", "A"))


In [ ]:
### for each entry in df-overlap, get a before and after sequence
### then do things with it
### count RGs lost (remember position of RGs before and compare with the after sequence) 

### make a column that is len(alt)/len(REF) and direction (F or R)
### check how many variants i have that are divisble by 3, where i can apply a 1to1 variant search



with open('genomic_coordinates_info_pos.json', 'r') as f:
    pos_genomic_coordinates_df = json.load(f)

for i,el in enumerate(pos_genomic_coordinates_df):
    curr_dict = pos_genomic_coordinates_df[i]
    curr_dict["region_id"] = str(el['protein'] + '_' + str(el['prot_region'][0]) + '_' + str(el['prot_region'][1]))
    pos_genomic_coordinates_df[i] = curr_dict
with open('genomic_coordinates_info_neg.json', 'r') as f:
    neg_genomic_coordinates_df = json.load(f)
for i,el in enumerate(neg_genomic_coordinates_df):
    curr_dict = neg_genomic_coordinates_df[i]
    curr_dict["region_id"] = str(el['protein'] + '_' + str(el['prot_region'][0]) + '_' + str(el['prot_region'][1]))
    neg_genomic_coordinates_df[i] = curr_dict

merged_genomic_coordinates_list = [
    {**d, "group": "pos"} for d in pos_genomic_coordinates_df
] + [
    {**d, "group": "neg"} for d in neg_genomic_coordinates_df
]

before_seq, after_seq = [], []
before_dna, after_dna = [], []
mut_pos = []

for i, el in df_overlap.iterrows():
    curr_region_id = el["region_id"]

    target = next((d for d in merged_genomic_coordinates_list if d["region_id"] == curr_region_id), None)
    # print(target)
    before_seq.append(target['prot_seq'])
    curr_before_dna = target['dna']
    before_dna.append(curr_before_dna)
    if len(target['intervals']) >1:
        # print("this one has mutliple cuts, lets work on this later")
        curr_transl_change = None
        curr_result = None
        curr_start_of_orig = None
    elif len(target['intervals']) == 1:
        ### best use case to work on
        curr_start_of_orig = el["Start"] - target['intervals'][0]["start"]
        curr_end_of_change = curr_start_of_orig + len(el["REF"])
        curr_dna_change = el["ALT"]
        curr_result = curr_before_dna[:curr_start_of_orig] + curr_dna_change + curr_before_dna[curr_end_of_change:]
        curr_transl_change = str(Seq(curr_result).translate())
        # print(curr_transl_change)
    else:
        print("This entry has 0 intervals!!!! How?? This should be impossible.")
    after_dna.append(curr_result)
    mut_pos.append(curr_start_of_orig)
    #### find position of the change
    # find the right interval 
    # right now skip ones that are overlapping in multiple
    # this minus the start of the interval is the start value, then exchange the old one to the new one and translate

    after_seq.append(curr_transl_change)
df_overlap["before_seq"] = before_seq
df_overlap["after_seq"] = after_seq

df_overlap["before_dna"] = before_dna
df_overlap["after_dna"] = after_dna


df_overlap["mut_pos"] = mut_pos


# out = df_overlap.apply(lambda row: get_RG_metrics(row.before_seq, row.after_seq), axis=1)
# print(out)
# Step 2 — convert each dict into columns
# out_df = pd.DataFrame(list(out))

# df_overlap = df_overlap.join(out)
# Step 1 — apply, returns a Series where each cell is a dict
# out = df_overlap.apply(
#     lambda row: get_RG_metrics(row.before_seq, row.after_seq),
#     axis=1
# )

df_overlap["variant_type"] = [
    classify_variant(row.before_dna, row.after_dna, row.before_seq, row.after_seq)
    for row in df_overlap.itertuples()
]


df_results = df_overlap.apply(
    lambda row: rg_change_from_category(
        category=row['variant_type'],
        before_aa=row['before_seq'],
        after_aa=row['after_seq'],
        mut_pos_dna=row['mut_pos'],
        ref_dna=row['REF'],
        alt_dna=row['ALT']
    ), axis=1
)

# df_results is a Series of dicts. Expand into separate columns
df_expanded = pd.json_normalize(df_results)

# Combine with original DataFrame
df_final = pd.concat([df_overlap, df_expanded], axis=1)

# print(df_final)


df_metrics = df_final.apply(
    lambda row: get_physchem_metrics(
        before_aa=row['before_seq'],
        after_aa=row['after_seq'],
        category = row['variant_type']
    ),
    axis=1
)

# Expand the resulting dictionaries into columns
df_metrics_expanded = pd.json_normalize(df_metrics)

# Combine with the original DataFrame
df_final2 = pd.concat([df_final, df_metrics_expanded], axis=1)

print(df_final2)
# out = df_overlap.apply(lambda r: rg_change_from_category(r.before_seq, r.after_seq), axis=1)
# empty = {k: None for k in out.dropna().iloc[0].keys()}
# out_df = out.apply(lambda x: x if isinstance(x, dict) else empty).apply(pd.Series)
# df_overlap = df_overlap.join(out_df)

# # Step 3 — join safely
# df_overlap = df_overlap.join(out_df)



In [ ]:
df_final2

In [ ]:
df_final2.to_pickle("variant_info_df.pkl")

In [ ]:
##### BACKUP 


import re
# as before we import the SequenceParameters class directly



# print(SeqParams("RRDDAARR").get_NCPR())
# print(SeqParams("RRDAA").get_FCR())
# print(SeqParams("RRDDAARR").get_mean_net_charge())
# print(SeqParams("RRDDYFWAARR").get_amino_acid_fractions()["Y"])

# def find_RG(seq: str):
#     return [m.start() for m in re.finditer(r"RG", seq)]

# def find_RGG(seq: str):
#     return [m.start() for m in re.finditer(r"RG", seq)]

# def charge_of(aa: str):
#     """Return numeric charge: +1, -1, 0. '*' and '-' return None."""
#     if aa in "KRH": return 1
#     if aa in "DE":  return -1
#     if aa in "-*":  return None
#     return 0

# def is_polar(aa: str):
#     # rough polar set: N, Q, S, T, C, Y
#     return aa in "NQSTCY"

# def is_aromatic(aa: str):
#     return aa in "FWYH"   # H included: aromatic ring

# def classify_RG_changes(before: str, after: str, ref: str, alt: str, mut_pos: int):
#     """
#     before  : original AA sequence
#     after   : mutated AA sequence
#     ref     : reference AAs replaced (e.g. 'QK')
#     alt     : replacement AAs (e.g. 'RG')
#     mut_pos : 0-based index of where REF begins in 'before'
#     """

#     # RG positions in both sequences
#     RG_before = find_RG(before)
#     RG_after = find_RG(after)

#     # Net length difference
#     Δ = len(alt) - len(ref)

#     # Convenient ranges
#     ref_start = mut_pos
#     ref_end = mut_pos + len(ref)     # exclusive
#     alt_end = mut_pos + len(alt)

#     untouched_RG = []
#     lost_RG = []
#     new_RG = []

#     # 1. RGs *before* mutation region → direct positional comparison
#     for pos in RG_before:
#         if pos + 1 < ref_start:  # RG spans positions pos,pos+1
#             if pos in RG_after:
#                 untouched_RG.append(pos)
#             else:
#                 lost_RG.append(pos)

#     # 2. RGs fully *inside* the mutation region → compare via coordinate mapping
#     for pos in RG_before:
#         if ref_start <= pos < ref_end:  
#             # Map position in 'before' to corresponding relative position in ALT
#             rel_pos = pos - ref_start
#             # Only count as untouched if ALT also has RG at this relative match
#             if 0 <= rel_pos < len(alt) - 1 and alt[rel_pos:rel_pos+2] == "RG":
#                 # The mapped absolute position in 'after'
#                 mapped_after_pos = ref_start + rel_pos
#                 if mapped_after_pos in RG_after:
#                     untouched_RG.append(pos)
#                 else:
#                     lost_RG.append(pos)
#             else:
#                 lost_RG.append(pos)

#     # 3. RGs *after* the mutation region → shifted positions
#     for pos in RG_before:
#         if pos >= ref_end:
#             mapped = pos + Δ
#             if mapped in RG_after:
#                 untouched_RG.append(pos)
#             else:
#                 lost_RG.append(pos)

#     # Now detect NEW RGs that didn't exist before
#     # Map after→before and see which have no corresponding origin
#     before_positions_possible = set()

#     # Reverse mapping for after positions
#     for pos in RG_after:
#         # Case: before mutation region
#         if pos + 1 < ref_start:
#             before_positions_possible.add(pos)

#         # Case: inside ALT region
#         elif ref_start <= pos < alt_end:
#             rel_pos = pos - ref_start
#             before_pos = ref_start + rel_pos
#             # only valid if inside original ref segment
#             if before_pos >= ref_start and before_pos < ref_end:
#                 before_positions_possible.add(before_pos)

#         # Case: after region, shift backwards
#         elif pos >= alt_end:
#             before_pos = pos - Δ
#             before_positions_possible.add(before_pos)

#     for pos in RG_after:
#         # A new RG is one whose mapped-before position wasn't an old RG position
#         mapped_candidates = []
#         # build same mapping as above quickly:
#         if pos + 1 < ref_start:
#             mapped_candidates = [pos]
#         elif ref_start <= pos < alt_end:
#             mapped_candidates = [ref_start + (pos - ref_start)]
#         else:
#             mapped_candidates = [pos - Δ]

#         if not any(c in RG_before for c in mapped_candidates):
#             new_RG.append(pos)

#     return {
#         "untouched_RG": sorted(list(set(untouched_RG))),
#         "lost_RG": sorted(list(set(lost_RG))),
#         "new_RG": sorted(list(set(new_RG))),
#     }
